# Notebook 2: Inspect, Iterate, Train

**Phases 3-4** | ~45 minutes | **The heart of the workshop**

In this notebook you will:
1. Load your auto-labeled dataset and inspect it programmatically
2. Run a student model and evaluate it against the teacher's labels
3. Analyze error distributions -- find WHERE the model fails
4. Compute mistakenness scores to find likely label errors
5. Visualize dataset structure with embeddings (ResNet18 + PCA/UMAP)
6. Curate the dataset (filter tiny labels) and compare before/after
7. Train a YOLO26n student model on curated labels
8. Evaluate the trained model and reflect

> **The data-centric insight**: The model is fine. The DATA is what you iterate on.
> Instead of tuning architectures or hyperparameters, we will find and fix specific
> data problems: wrong labels, redundant images, size-dependent failures.

> **AI assistance**: Use your preferred AI coding tool for guidance.
> See `cv_copilot_skill.md` for a ready-to-use system prompt.

---
## Section 0: Setup & Load Dataset

In [ ]:
# ── Cell 0a: Environment + Path Setup ─────────────────────────────────────────────────
import sys
import glob
import warnings
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yaml

warnings.filterwarnings("ignore", category=UserWarning)

# ── Device ───────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"Device:  {DEVICE}")

# ── Paths ────────────────────────────────────────────────────────────────────
WORKSHOP_DIR = Path(".").resolve().parent
DATA_DIR = WORKSHOP_DIR / "data"

# ============================================================
# CONFIGURE: Point this to YOUR labeled dataset from Notebook 1
# ============================================================
DATASET_DIR = DATA_DIR / "ppe_dataset_sam3_exp_a"  # <-- adjust if you used a different name
CLASS_NAMES = {0: "hardhat", 1: "person"}

# Find trained weights (from a previous training run, if any)
runs_dir = DATA_DIR / "ppe_results"
run_dirs = sorted(glob.glob(str(runs_dir / "*/weights/best.pt")))
BEST_WEIGHTS = Path(run_dirs[-1]) if run_dirs else None

print(f"\nWorkshop dir:  {WORKSHOP_DIR}")
print(f"Dataset dir:   {DATASET_DIR}")
print(f"Best weights:  {BEST_WEIGHTS}")
print(f"Classes:       {CLASS_NAMES}")

assert DATASET_DIR.exists(), f"Dataset not found at {DATASET_DIR} -- run Notebook 1 first!"

In [ ]:
# ── Cell 0b: Helper Functions ──────────────────────────────────────────────
# These utilities are used throughout this notebook. Run this cell once now.

from PIL import Image
from collections import defaultdict
import re

# ── YOLO label parsing & box conversion ──────────────────────────────────

def parse_yolo_labels(label_path):
    """Parse YOLO label file -> list of (cls_id, cx, cy, w, h)."""
    labels = []
    p = Path(label_path)
    if not p.exists():
        return labels
    for line in p.read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        try:
            cls_id = int(parts[0])
            cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            labels.append((cls_id, cx, cy, w, h))
        except ValueError:
            continue
    return labels


def yolo_to_xyxy(cx, cy, w, h, img_w, img_h):
    """Convert YOLO normalized (cx, cy, w, h) to pixel [x1, y1, x2, y2]."""
    x1 = (cx - w / 2) * img_w
    y1 = (cy - h / 2) * img_h
    x2 = (cx + w / 2) * img_w
    y2 = (cy + h / 2) * img_h
    return [x1, y1, x2, y2]


def compute_iou(box1, box2):
    """IoU between two [x1, y1, x2, y2] boxes."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0.0 else 0.0


def match_predictions(gt_boxes, pred_boxes, iou_threshold=0.5):
    """Match predictions to GT boxes using greedy IoU matching.

    Args:
        gt_boxes: list of [x1, y1, x2, y2] ground truth boxes (single class)
        pred_boxes: list of dicts with keys 'bbox' ([x1,y1,x2,y2]), 'conf', 'cls'
                    sorted by confidence (highest first)
        iou_threshold: minimum IoU for a match

    Returns:
        tp, fp, fn counts
    """
    matched_gt = set()
    tp = 0
    fp = 0

    for pred in pred_boxes:
        best_iou = 0.0
        best_gt_idx = -1
        for i, gt_box in enumerate(gt_boxes):
            if i in matched_gt:
                continue
            ov = compute_iou(pred["bbox"], gt_box)
            if ov > best_iou:
                best_iou = ov
                best_gt_idx = i
        if best_iou >= iou_threshold and best_gt_idx >= 0:
            tp += 1
            matched_gt.add(best_gt_idx)
        else:
            fp += 1

    fn = len(gt_boxes) - len(matched_gt)
    return tp, fp, fn


def evaluate_image(gt_labels, pred_boxes, img_w, img_h, class_names, iou_threshold=0.5):
    """Evaluate predictions vs GT for a single image, per class.

    Args:
        gt_labels: list of (cls_id, cx, cy, w, h) from parse_yolo_labels()
        pred_boxes: list of dicts {'cls': int, 'bbox': [x1,y1,x2,y2], 'conf': float}
        img_w, img_h: image dimensions in pixels
        class_names: dict {cls_id: name}
        iou_threshold: IoU threshold for matching

    Returns:
        dict with keys 'tp', 'fp', 'fn' (totals) and 'per_class' breakdown
    """
    # Group GT boxes by class
    gt_by_class = defaultdict(list)
    for cls_id, cx, cy, w, h in gt_labels:
        box = yolo_to_xyxy(cx, cy, w, h, img_w, img_h)
        gt_by_class[cls_id].append(box)

    # Group predictions by class, sorted by confidence (descending)
    pred_by_class = defaultdict(list)
    for pred in pred_boxes:
        pred_by_class[pred["cls"]].append(pred)
    for cls_id in pred_by_class:
        pred_by_class[cls_id].sort(key=lambda x: -x["conf"])

    all_classes = set(gt_by_class.keys()) | set(pred_by_class.keys())

    total_tp, total_fp, total_fn = 0, 0, 0
    per_class = {}

    for cls_id in all_classes:
        gt_cls = gt_by_class.get(cls_id, [])
        pred_cls = pred_by_class.get(cls_id, [])
        tp, fp, fn = match_predictions(gt_cls, pred_cls, iou_threshold)
        total_tp += tp
        total_fp += fp
        total_fn += fn
        per_class[cls_id] = {"tp": tp, "fp": fp, "fn": fn}

    return {"tp": total_tp, "fp": total_fp, "fn": total_fn, "per_class": per_class}


# ── Dataset loading helpers ──────────────────────────────────────────────

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def load_dataset_split(dataset_dir, split="val", class_names=None):
    """Load a YOLO dataset split into a list of sample dicts.

    Each sample dict has:
        - filepath: str (absolute path to image)
        - stem: str (filename without extension)
        - gt_labels: list of (cls_id, cx, cy, w, h)
        - img_w, img_h: int (image dimensions)

    Args:
        dataset_dir: Path to YOLO dataset root
        split: 'train' or 'val'
        class_names: dict {cls_id: name} (optional, stored on each sample)
    """
    dataset_dir = Path(dataset_dir)
    images_dir = dataset_dir / "images" / split
    labels_dir = dataset_dir / "labels" / split

    assert images_dir.exists(), f"Images dir not found: {images_dir}"

    samples = []
    image_files = sorted(
        p for p in images_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    )

    for img_path in image_files:
        stem = img_path.stem
        label_path = labels_dir / f"{stem}.txt"
        gt_labels = parse_yolo_labels(label_path)

        img = Image.open(img_path)
        img_w, img_h = img.size
        img.close()

        samples.append({
            "filepath": str(img_path),
            "stem": stem,
            "gt_labels": gt_labels,
            "img_w": img_w,
            "img_h": img_h,
            "class_names": class_names or CLASS_NAMES,
        })

    return samples


# ── Visualization ────────────────────────────────────────────────────────

def plot_gt_vs_pred(sample, figsize=(16, 6)):
    """Plot ground truth and predictions side by side for one sample dict.

    Args:
        sample: dict with keys 'filepath', 'gt_labels', 'predictions',
                'img_w', 'img_h', 'class_names', and optionally
                'eval_tp', 'eval_fp', 'eval_fn'
    """
    class_colors = {
        "hardhat": "#27ae60", "person": "#3498db",
        "bottle": "#27ae60", "cap": "#3498db",
    }
    class_names = sample.get("class_names", CLASS_NAMES)

    img = np.array(Image.open(sample["filepath"]))
    h, w = img.shape[:2]

    fig, (ax_gt, ax_pred) = plt.subplots(1, 2, figsize=figsize)

    # --- Ground truth (from YOLO labels) ---
    ax_gt.imshow(img)
    gt_labels = sample.get("gt_labels", [])
    for cls_id, cx, cy, bw, bh in gt_labels:
        x1, y1, x2, y2 = yolo_to_xyxy(cx, cy, bw, bh, w, h)
        name = class_names.get(cls_id, f"cls_{cls_id}")
        color = class_colors.get(name, "#e74c3c")
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                              linewidth=2, edgecolor=color, facecolor="none")
        ax_gt.add_patch(rect)
        ax_gt.text(x1, y1 - 3, name,
                   color="white", fontsize=7, fontweight="bold",
                   bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8))
    ax_gt.set_title(f"Ground Truth (teacher labels)\n({len(gt_labels)} detections)", fontsize=10)
    ax_gt.axis("off")

    # --- Predictions ---
    ax_pred.imshow(img)
    predictions = sample.get("predictions", [])
    for pred in predictions:
        x1, y1, x2, y2 = pred["bbox"]
        cls_id = pred["cls"]
        conf = pred.get("conf", None)
        eval_tag = pred.get("eval", None)

        name = class_names.get(cls_id, f"cls_{cls_id}")
        color = class_colors.get(name, "#e74c3c")
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                              linewidth=2, edgecolor=color, facecolor="none")
        ax_pred.add_patch(rect)
        conf_str = f" {conf:.2f}" if conf is not None else ""
        eval_str = f" [{eval_tag}]" if eval_tag else ""
        ax_pred.text(x1, y1 - 3, f"{name}{conf_str}{eval_str}",
                     color="white", fontsize=7, fontweight="bold",
                     bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8))
    ax_pred.set_title(f"Predictions (student model)\n({len(predictions)} detections)", fontsize=10)
    ax_pred.axis("off")

    tp = sample.get("eval_tp", 0)
    fp = sample.get("eval_fp", 0)
    fn = sample.get("eval_fn", 0)
    fig.suptitle(
        f"{Path(sample['filepath']).name}   |   TP={tp}  FP={fp}  FN={fn}",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()


# ── Scene category extraction ────────────────────────────────────────────

def extract_category(stem):
    """Extract scene category from a filename stem like 'cctv_cctv_01'.

    Strategy: remove the trailing _NN number, then check if the
    remaining string is a doubled category (cat_cat). If the prefix
    has the form X_X, return X.
    """
    m = re.match(r"^(.+)_(\d+)$", stem)
    if not m:
        return "unknown"
    prefix = m.group(1)

    parts = prefix.split("_")
    for i in range(1, len(parts)):
        left = "_".join(parts[:i])
        right = "_".join(parts[i:])
        if left == right:
            return left
    return prefix


print("Helper functions loaded:")
print("  parse_yolo_labels(), yolo_to_xyxy(), compute_iou(), match_predictions()")
print("  evaluate_image(), load_dataset_split(), plot_gt_vs_pred()")
print("  extract_category()")

In [ ]:
# ── Cell 0c: Load YOLO dataset directly ─────────────────────────────────
# Read the validation split by parsing image files and YOLO label files.
# No database or external tool needed — just file I/O.

yaml_path = DATASET_DIR / "dataset.yaml"
if not yaml_path.exists():
    yaml_path = DATASET_DIR / "data.yaml"
assert yaml_path.exists(), f"YAML not found at {DATASET_DIR} -- check your dataset!"

# Read class names from dataset.yaml
with open(yaml_path) as f:
    ds_config = yaml.safe_load(f)
yaml_class_names = ds_config.get("names", CLASS_NAMES)
if isinstance(yaml_class_names, list):
    yaml_class_names = {i: n for i, n in enumerate(yaml_class_names)}
print(f"Classes from YAML: {yaml_class_names}")

# Load val split
dataset = load_dataset_split(DATASET_DIR, split="val", class_names=CLASS_NAMES)

print(f"\nDataset loaded from: {DATASET_DIR}")
print(f"Validation samples: {len(dataset)}")

# Count ground truth labels by class
class_counts = defaultdict(int)
for sample in dataset:
    for cls_id, *_ in sample["gt_labels"]:
        class_counts[cls_id] += 1

print(f"\nGround truth labels (from teacher):")
for cls_id in sorted(class_counts.keys()):
    cls_name = CLASS_NAMES.get(cls_id, f"cls_{cls_id}")
    print(f"  {cls_name:15s}  {class_counts[cls_id]:4d} instances")
print(f"  {'TOTAL':15s}  {sum(class_counts.values()):4d} instances")

---
## Section 1: Run Student Model & Evaluate

We load a trained baseline model (or use pre-baked weights) and run inference on every
validation image. We store predictions alongside ground truth -- same images,
two sets of bounding boxes -- so we can compare them per-sample.

After evaluation, every sample gets three new fields:
- **eval_tp**: True Positives (student agrees with teacher)
- **eval_fp**: False Positives (student detects something the teacher did not label)
- **eval_fn**: False Negatives (teacher labeled something the student missed)

In [ ]:
# ── Cell 1a: Add student model predictions ────────────────────────────────
from ultralytics import YOLO

# Load trained student model
assert BEST_WEIGHTS is not None and Path(BEST_WEIGHTS).exists(), (
    f"No trained weights found at {BEST_WEIGHTS}. "
    f"Either train a model first (Section 8) or check data/ppe_results/ for pre-baked weights."
)

student = YOLO(str(BEST_WEIGHTS))
print(f"Loaded student model from: {BEST_WEIGHTS}")
print(f"Model classes: {student.names}")

# Run inference on every sample and store predictions
print(f"\nRunning inference on {len(dataset)} images...")
pred_counts = defaultdict(int)

for i, sample in enumerate(dataset):
    results = student.predict(sample["filepath"], verbose=False, device=DEVICE)
    preds = []
    if results and results[0].boxes is not None:
        for box in results[0].boxes:
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            xyxy = box.xyxy[0].tolist()
            preds.append({"cls": cls_id, "bbox": xyxy, "conf": conf})
            cls_name = CLASS_NAMES.get(cls_id, f"cls_{cls_id}")
            pred_counts[cls_name] += 1
    sample["predictions"] = preds

    if (i + 1) % 20 == 0 or (i + 1) == len(dataset):
        print(f"  {i+1}/{len(dataset)} images processed")

print(f"\nStudent predictions:")
for cls_name, count in sorted(pred_counts.items()):
    print(f"  {cls_name:15s}  {count:4d} instances")

gt_total = sum(class_counts.values())
pred_total = sum(pred_counts.values())
print(f"\nGround truth total: {gt_total}")
print(f"Prediction total:   {pred_total}")
print(f"Difference:         {pred_total - gt_total:+d}")

In [ ]:
# ── Cell 1b: Run IoU-based evaluation (TP/FP/FN matching) ────────────────
# We match each prediction to ground truth using IoU >= 0.5,
# then compute per-class precision, recall, and F1.

IOU_THRESHOLD = 0.50

# Aggregate per-class metrics
agg_metrics = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

for sample in dataset:
    eval_result = evaluate_image(
        sample["gt_labels"], sample["predictions"],
        sample["img_w"], sample["img_h"],
        CLASS_NAMES, iou_threshold=IOU_THRESHOLD,
    )
    sample["eval_tp"] = eval_result["tp"]
    sample["eval_fp"] = eval_result["fp"]
    sample["eval_fn"] = eval_result["fn"]
    sample["eval_per_class"] = eval_result["per_class"]

    for cls_id, counts in eval_result["per_class"].items():
        agg_metrics[cls_id]["tp"] += counts["tp"]
        agg_metrics[cls_id]["fp"] += counts["fp"]
        agg_metrics[cls_id]["fn"] += counts["fn"]

# Compute overall metrics
total_tp = sum(m["tp"] for m in agg_metrics.values())
total_fp = sum(m["fp"] for m in agg_metrics.values())
total_fn = sum(m["fn"] for m in agg_metrics.values())
overall_prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
overall_rec = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
overall_f1 = 2 * overall_prec * overall_rec / (overall_prec + overall_rec) if (overall_prec + overall_rec) > 0 else 0

# Also run Ultralytics model.val() for official mAP
print("Running Ultralytics model.val() for official mAP...")
val_results = student.val(
    data=str(yaml_path),
    device=DEVICE,
    verbose=False,
)
baseline_mAP50 = float(val_results.box.map50)
baseline_mAP50_95 = float(val_results.box.map)

print("=" * 60)
print(f"  mAP@0.5 = {baseline_mAP50:.4f}   (Ultralytics official)")
print(f"  mAP@0.5:0.95 = {baseline_mAP50_95:.4f}")
print("=" * 60)

print(f"\nPer-class evaluation (IoU >= {IOU_THRESHOLD}):")
print("-" * 70)
print(f"  {'Class':<15} {'TP':>6} {'FP':>6} {'FN':>6} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"  {'-'*15} {'-'*6} {'-'*6} {'-'*6} {'-'*10} {'-'*10} {'-'*10}")

for cls_id in sorted(agg_metrics.keys()):
    m = agg_metrics[cls_id]
    tp, fp, fn = m["tp"], m["fp"], m["fn"]
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    cls_name = CLASS_NAMES.get(cls_id, f"cls_{cls_id}")
    print(f"  {cls_name:<15} {tp:>6} {fp:>6} {fn:>6} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}")

print(f"  {'-'*15} {'-'*6} {'-'*6} {'-'*6} {'-'*10} {'-'*10} {'-'*10}")
print(f"  {'TOTAL':<15} {total_tp:>6} {total_fp:>6} {total_fn:>6} {overall_prec:>10.4f} {overall_rec:>10.4f} {overall_f1:>10.4f}")

### Think about it

- Which class has worse **recall**? What does low recall mean in practice for a safety system?
- If `person` recall is 90% but `hardhat` recall is 60%, what happens to compliance decisions?
- Is mAP the right metric for a safety application, or should we care more about recall?

---
## Section 2: Error Distribution Analysis

A single mAP number hides critical information. We need to understand:
- Are errors spread evenly across images, or concentrated in a few?
- Which specific images are the worst?
- When the student and teacher disagree, who is right?

### 2.1 TP / FP / FN Histograms

> **Think about it**: Before running this cell, predict: will errors be uniform
> across images, or will a few images dominate? What would each pattern imply
> for your data improvement strategy?

In [ ]:
# ── Cell 2a: Per-image error histograms ───────────────────────────────────

tp_values = [s["eval_tp"] for s in dataset]
fp_values = [s["eval_fp"] for s in dataset]
fn_values = [s["eval_fn"] for s in dataset]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, values, title, color in zip(
    axes,
    [tp_values, fp_values, fn_values],
    ["True Positives", "False Positives", "False Negatives"],
    ["#27ae60", "#e74c3c", "#f39c12"],
):
    max_val = max(values) if values else 0
    ax.hist(values, bins=range(0, max_val + 2), color=color, edgecolor="black", alpha=0.8)
    ax.set_xlabel("Count per image")
    ax.set_ylabel("Number of images")
    ax.set_title(f"{title}\n(mean={np.mean(values):.1f}, max={max_val})")
    ax.grid(True, alpha=0.3)

plt.suptitle("Per-Image Detection Results", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Think about it

- Are errors spread evenly or concentrated in a few images?
- What does a **long FP tail** tell you? (Hint: the student detects objects the teacher never labeled.)
- What does a **long FN tail** tell you? (Hint: the teacher labeled objects the student cannot find.)

### 2.2 Worst-Case Identification

> **Think about it**: Look at the filenames of the worst images below.
> Do they share visual characteristics? (scene type, camera angle, crowd density?)

In [ ]:
# ── Cell 2b: Top-5 worst images by FP and FN ──────────────────────────────

print("Top 5 images with most FALSE POSITIVES:")
print("  (Student detects objects the teacher did NOT label)")
print("  Could be hallucinations OR teacher omissions -- check visually below.\n")
worst_fp = sorted(dataset, key=lambda s: s["eval_fp"], reverse=True)
for i, sample in enumerate(worst_fp[:5]):
    n_gt = len(sample["gt_labels"])
    n_pred = len(sample["predictions"])
    print(f"  {i+1}. FP={sample['eval_fp']}, FN={sample['eval_fn']}, TP={sample['eval_tp']}  "
          f"(GT={n_gt}, Pred={n_pred})  |  {Path(sample['filepath']).name}")

print(f"\nTop 5 images with most FALSE NEGATIVES:")
print("  (Teacher labeled objects the student MISSED)\n")
worst_fn = sorted(dataset, key=lambda s: s["eval_fn"], reverse=True)
for i, sample in enumerate(worst_fn[:5]):
    n_gt = len(sample["gt_labels"])
    n_pred = len(sample["predictions"])
    print(f"  {i+1}. FN={sample['eval_fn']}, FP={sample['eval_fp']}, TP={sample['eval_tp']}  "
          f"(GT={n_gt}, Pred={n_pred})  |  {Path(sample['filepath']).name}")

### 2.3 Visualize the Worst Failures

> **Think about it**: For each image below, decide:
> 1. Is the **student** wrong? (missed a real object or hallucinated one)
> 2. Is the **teacher** wrong? (mislabeled or missed an object in the auto-labels)
> 3. If many FPs turn out to be real objects the teacher missed, what does that
>    tell you about the quality of your auto-labeler?

In [ ]:
# ── Cell 2c: Visualize 4 worst images by total errors ────────────────────

error_sorted = sorted(dataset, key=lambda s: s["eval_fp"] + s["eval_fn"], reverse=True)

print("Worst 4 images by total detection errors (FP + FN):\n")
for sample in error_sorted[:4]:
    plot_gt_vs_pred(sample)

---
## Section 3: Label Quality Analysis

Standard evaluation tells us where the **student** fails. But in a distillation pipeline,
the ground truth came from an auto-labeler (the **teacher**). When the student disagrees
with the teacher, who is right?

Our **mistakenness** score answers this question:
- For each image, we compute `(FP + FN) / (TP + FP + FN)` -- the disagreement rate
- If the student confidently disagrees on many detections, the teacher is likely wrong
- High mistakenness = likely annotation error from the auto-labeler

This is the core data-centric insight: **use the student to audit the teacher.**

### 3.1 Compute Mistakenness

> **Think about it**: The mistakenness score uses the student to audit the teacher.
> If the student **confidently** disagrees with a label, who is more likely wrong --
> the student (trained on hundreds of examples) or the teacher (ran once per image)?

We compute a simple disagreement proxy: `(FP + FN) / (TP + FP + FN)` per image.
A score of 0 means perfect agreement; 1 means total disagreement.

In [ ]:
# ── Cell 3a: Compute mistakenness scores ─────────────────────────────────
# Mistakenness = (FP + FN) / (TP + FP + FN) per image.
# This measures how much the student disagrees with the teacher's labels.
# High score = likely annotation error from the auto-labeler.

print("Computing mistakenness (likelihood that teacher labels are wrong)...")
print("This compares student predictions vs teacher labels per-sample.\n")

for sample in dataset:
    tp = sample["eval_tp"]
    fp = sample["eval_fp"]
    fn = sample["eval_fn"]
    total = tp + fp + fn
    sample["mistakenness"] = (fp + fn) / total if total > 0 else 0.0

# Plot distribution
mistake_vals = [s["mistakenness"] for s in dataset]
p90 = np.percentile(mistake_vals, 90)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(mistake_vals, bins=30, color="#e74c3c", edgecolor="black", alpha=0.8)
ax.set_xlabel("Mistakenness Score")
ax.set_ylabel("Number of Images")
ax.set_title("Label Mistakenness Distribution\n(High = likely annotation error from teacher)")
ax.axvline(x=p90, color="#2c3e50", linestyle="--", linewidth=2,
           label=f"90th percentile = {p90:.3f}")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Top 10 most suspicious samples
print("\nTop 10 most SUSPICIOUS samples (highest mistakenness):")
print("These are where the student CONFIDENTLY DISAGREES with the teacher.\n")
suspicious = sorted(dataset, key=lambda s: s["mistakenness"], reverse=True)
for i, sample in enumerate(suspicious[:10]):
    tp = sample["eval_tp"]
    fp = sample["eval_fp"]
    fn = sample["eval_fn"]
    print(f"  {i+1:2d}. mistakenness={sample['mistakenness']:.3f}  |  "
          f"TP={tp} FP={fp} FN={fn}  |  {Path(sample['filepath']).name}")

### 3.2 Visualize Suspicious Samples

> **Think about it**: For each suspicious image below, make a judgment call:
> 1. **Fix the teacher's label** -- the auto-labeler got it wrong
> 2. **Student is wrong** -- the auto-label was correct, model is confused
> 3. **Ambiguous** -- hard to tell even for a human
>
> Keep a mental tally of your decisions. If most are (1), your auto-labeler needs work.

In [ ]:
# ── Cell 3b: Visualize top-3 suspicious samples ──────────────────────────

print("Top 3 most suspicious samples -- are the teacher's labels correct?\n")
suspicious = sorted(dataset, key=lambda s: s["mistakenness"], reverse=True)
for sample in suspicious[:3]:
    plot_gt_vs_pred(sample)

---
## Section 4: Data Diversity Analysis

Are there redundant images in our dataset? Images that are near-duplicates add training
cost without adding information. A **uniqueness** score identifies them.

We extract image features using a pretrained ResNet18 backbone, then:
- Compute pairwise cosine distances to derive uniqueness scores
- Reduce features to 2D with PCA or UMAP to visualize dataset structure --
  clusters, outliers, and gaps

### 4.1 Compute Image Embeddings & Uniqueness

We use a pretrained ResNet18 to extract a 512-dimensional feature vector per image,
then compute uniqueness as the average cosine distance to the K nearest neighbors.
Low uniqueness = many similar images nearby = redundant.

> **Think about it**: If 20% of your dataset is near-duplicate, would removing them
> hurt or help? Form a hypothesis before looking at the results.

In [ ]:
# ── Cell 4a: Extract features and compute uniqueness ─────────────────────
import torchvision.models as models
import torchvision.transforms as T
from torch.nn.functional import cosine_similarity

print("Loading pretrained ResNet18 for feature extraction...")
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.eval()

# Remove the final classification layer to get 512-d features
feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1])
feature_extractor = feature_extractor.to(DEVICE)
feature_extractor.eval()

# Standard ImageNet preprocessing
preprocess = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"Extracting features from {len(dataset)} images...")
all_features = []

with torch.no_grad():
    for i, sample in enumerate(dataset):
        img = Image.open(sample["filepath"]).convert("RGB")
        tensor = preprocess(img).unsqueeze(0).to(DEVICE)
        feat = feature_extractor(tensor).squeeze()  # shape: (512,)
        all_features.append(feat.cpu())
        if (i + 1) % 20 == 0 or (i + 1) == len(dataset):
            print(f"  {i+1}/{len(dataset)} images processed")

# Stack into matrix (N, 512) and normalize
features = torch.stack(all_features)
features_norm = features / features.norm(dim=1, keepdim=True)

# Compute pairwise cosine similarity matrix
print("\nComputing pairwise cosine similarities...")
sim_matrix = features_norm @ features_norm.T  # (N, N)

# Uniqueness = average cosine distance to K nearest neighbors
K = min(5, len(dataset) - 1)
sim_np = sim_matrix.numpy()
np.fill_diagonal(sim_np, -1)  # exclude self-similarity

for sample_idx in range(len(dataset)):
    # Get top-K most similar (highest cosine similarity)
    top_k_sim = np.sort(sim_np[sample_idx])[-K:]
    # Uniqueness = 1 - mean similarity to nearest neighbors
    dataset[sample_idx]["uniqueness"] = float(1.0 - np.mean(top_k_sim))

# Plot distribution
uniqueness_vals = [s["uniqueness"] for s in dataset]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(uniqueness_vals, bins=30, color="#3498db", edgecolor="black", alpha=0.8)
ax.set_xlabel("Uniqueness Score")
ax.set_ylabel("Number of Images")
ax.set_title("Image Uniqueness Distribution\n(Low = redundant, High = distinct)")
ax.axvline(x=np.median(uniqueness_vals), color="#e74c3c", linestyle="--",
           label=f"Median = {np.median(uniqueness_vals):.3f}")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 5 most redundant
print("\n5 most REDUNDANT images (candidates for removal):")
redundant = sorted(dataset, key=lambda s: s["uniqueness"])
for i, sample in enumerate(redundant[:5]):
    print(f"  {i+1}. uniqueness={sample['uniqueness']:.3f}  |  {Path(sample['filepath']).name}")

# 5 most unique
print("\n5 most UNIQUE images (highest information value):")
unique = sorted(dataset, key=lambda s: s["uniqueness"], reverse=True)
for i, sample in enumerate(unique[:5]):
    print(f"  {i+1}. uniqueness={sample['uniqueness']:.3f}  |  {Path(sample['filepath']).name}")

### 4.2 2D Embeddings Visualization

We reduce the ResNet18 feature vectors from 512 to 2 dimensions using PCA (or UMAP
if installed). This reveals:
- **Clusters**: groups of visually similar images (same scene type)
- **Outliers**: images very different from everything else
- **Distribution gaps**: visual conditions with no training data

> **Think about it**: Are errors spatially clustered? Do redundant images overlap
> with error-prone regions? If so, what does that tell you about what data to fix?

In [ ]:
# ── Cell 4b: Compute 2D embeddings ────────────────────────────────────────
from sklearn.decomposition import PCA

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("umap-learn not installed. Falling back to PCA.")
    print("Install with: pip install umap-learn\n")

features_np = features_norm.numpy()

if HAS_UMAP:
    viz_method = "umap"
    print("Computing 2D embeddings with UMAP...")
    reducer = umap.UMAP(n_components=2, random_state=51, n_neighbors=min(15, len(dataset) - 1))
    coords = reducer.fit_transform(features_np)
else:
    viz_method = "pca"
    print("Computing 2D embeddings with PCA...")
    pca = PCA(n_components=2, random_state=51)
    coords = pca.fit_transform(features_np)
    print(f"  PCA explained variance: {pca.explained_variance_ratio_[0]:.2%} + {pca.explained_variance_ratio_[1]:.2%}")

print(f"Visualization computed. {len(dataset)} points in 2D.")

In [ ]:
# ── Cell 4c: Three scatter plots colored by different metrics ───────────────

assert coords.shape == (len(dataset), 2), f"Expected ({len(dataset)}, 2), got {coords.shape}"

uniqueness = np.array([s["uniqueness"] for s in dataset])
mistakenness = np.array([s["mistakenness"] for s in dataset])
total_errors = np.array([s["eval_fp"] for s in dataset]) + np.array([s["eval_fn"] for s in dataset])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Colored by uniqueness
sc1 = axes[0].scatter(coords[:, 0], coords[:, 1], c=uniqueness,
                       cmap="RdYlGn", s=60, edgecolors="black", linewidths=0.3, alpha=0.8)
axes[0].set_title("Colored by Uniqueness\n(green=unique, red=redundant)")
plt.colorbar(sc1, ax=axes[0], label="Uniqueness")

# Plot 2: Colored by mistakenness
sc2 = axes[1].scatter(coords[:, 0], coords[:, 1], c=mistakenness,
                       cmap="RdYlGn_r", s=60, edgecolors="black", linewidths=0.3, alpha=0.8)
axes[1].set_title("Colored by Mistakenness\n(red=likely label error)")
plt.colorbar(sc2, ax=axes[1], label="Mistakenness")

# Plot 3: Colored by total errors
sc3 = axes[2].scatter(coords[:, 0], coords[:, 1], c=total_errors,
                       cmap="YlOrRd", s=60, edgecolors="black", linewidths=0.3, alpha=0.8)
axes[2].set_title("Colored by Total Errors (FP+FN)\n(dark=more errors)")
plt.colorbar(sc3, ax=axes[2], label="FP + FN")

for ax in axes:
    ax.set_xlabel(f"{viz_method.upper()}-1")
    ax.set_ylabel(f"{viz_method.upper()}-2")
    ax.grid(True, alpha=0.2)

plt.suptitle("Dataset Embeddings -- Three Perspectives", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Look for spatial patterns:")
print("  - Are errors clustered in one region? (= specific visual conditions cause failures)")
print("  - Are redundant images clustered? (= similar images from the same scene type)")
print("  - Are label errors near the boundary between clusters?")

---
## Section 5: Data Quality Dashboard

Let us combine all the metrics into a single dashboard view. This shows the
relationship between label quality (mistakenness), detection errors (FP+FN),
and data redundancy (uniqueness).

> **Think about it**: If you could only fix **5 images** before re-training,
> which 5 would you choose and why? (Top-right corner of the first plot = highest priority.)

In [ ]:
# ── Cell 5a: Combined quality scatter plots ───────────────────────────────

errors = np.array([s["eval_fp"] + s["eval_fn"] for s in dataset])
mistake = np.array([s["mistakenness"] for s in dataset])
uniq = np.array([s["uniqueness"] for s in dataset])
names = [Path(s["filepath"]).name for s in dataset]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Mistakenness vs Total Errors
ax = axes[0]
sc = ax.scatter(mistake, errors, c=uniq, cmap="RdYlGn", s=80,
                edgecolors="black", linewidths=0.5, alpha=0.8)
plt.colorbar(sc, ax=ax, label="Uniqueness")
ax.set_xlabel("Mistakenness", fontsize=11)
ax.set_ylabel("Total Errors (FP + FN)", fontsize=11)
ax.set_title("Label Quality Map\n(top-right = worst samples)")

# Annotate the worst points
worst_idx = np.argsort(mistake + errors / max(errors.max(), 1))[-5:]
for idx in worst_idx:
    ax.annotate(names[idx], (mistake[idx], errors[idx]),
                fontsize=7, alpha=0.8,
                textcoords="offset points", xytext=(5, 5))
ax.grid(True, alpha=0.3)

# Plot 2: Uniqueness vs Mistakenness
ax2 = axes[1]
sc2 = ax2.scatter(uniq, mistake, c=errors, cmap="YlOrRd", s=80,
                  edgecolors="black", linewidths=0.5, alpha=0.8)
plt.colorbar(sc2, ax=ax2, label="Total Errors")
ax2.set_xlabel("Uniqueness", fontsize=11)
ax2.set_ylabel("Mistakenness", fontsize=11)
ax2.set_title("Redundancy vs Label Quality\n(top-left = redundant AND wrong labels)")
ax2.grid(True, alpha=0.3)

plt.suptitle("Data Quality Dashboard", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary statistics
n_total = len(dataset)
n_perfect = int(np.sum(errors == 0))
n_high_mistake = int(np.sum(mistake > np.percentile(mistake, 80)))
n_low_unique = int(np.sum(uniq < np.percentile(uniq, 20)))

print(f"\n{'='*60}")
print(f"  DATA QUALITY SUMMARY")
print(f"{'='*60}")
print(f"  Total validation images:          {n_total}")
print(f"  Perfect agreement (0 errors):     {n_perfect} ({n_perfect/n_total*100:.0f}%)")
print(f"  High mistakenness (top 20%):      {n_high_mistake} images -- review labels")
print(f"  Low uniqueness (bottom 20%):      {n_low_unique} images -- consider removing")
print(f"  mAP@0.5 (official):               {baseline_mAP50:.4f}")
print(f"{'='*60}")

---
## Section 6: Per-Scene Category Analysis

Our synthetic PPE dataset has images from different scene categories (cctv, warehouse,
highway, etc.). The scene category is encoded in the filename: `cctv_cctv_01.jpg`
has category `cctv`.

Let us check if the model performs differently across scene types.

> **Think about it**: Which scene type do you expect to have the **worst** performance?
> (Think about object size, occlusion, camera distance, lighting.)

In [ ]:
# ── Cell 6a: Extract scene category from filenames ───────────────────────

# Parse scene category from filenames like "cctv_cctv_01.jpg" -> "cctv"
# or "mixed_compliance_mixed_compliance_05.jpg" -> "mixed_compliance"
# using the extract_category() helper defined in cell 0b.

for sample in dataset:
    stem = Path(sample["filepath"]).stem
    sample["scene_category"] = extract_category(stem)

# Show categories found
categories = sorted(set(s["scene_category"] for s in dataset))
print(f"Scene categories found: {categories}")
print(f"\nImages per category:")
for cat in categories:
    count = sum(1 for s in dataset if s["scene_category"] == cat)
    print(f"  {cat:25s}  {count:3d} images")

In [ ]:
# ── Cell 6b: Per-category error summary ──────────────────────────────────
# We compute per-category precision, recall, F1, and average errors.
# (mAP per-category would require AP curve computation -- we use P/R/F1 as a simpler
# but still informative metric. The official mAP from model.val() is computed once above.)

print(f"{'Category':25s} {'Images':>7s} {'Precision':>10s} {'Recall':>10s} {'F1':>8s} {'Avg FP':>8s} {'Avg FN':>8s} {'Avg Mistake':>12s}")
print("-" * 95)

cat_stats = []
for cat in sorted(categories):
    cat_samples = [s for s in dataset if s["scene_category"] == cat]
    n_imgs = len(cat_samples)

    # Aggregate TP/FP/FN for this category
    cat_tp = sum(s["eval_tp"] for s in cat_samples)
    cat_fp = sum(s["eval_fp"] for s in cat_samples)
    cat_fn = sum(s["eval_fn"] for s in cat_samples)
    cat_prec = cat_tp / (cat_tp + cat_fp) if (cat_tp + cat_fp) > 0 else 0
    cat_rec = cat_tp / (cat_tp + cat_fn) if (cat_tp + cat_fn) > 0 else 0
    cat_f1 = 2 * cat_prec * cat_rec / (cat_prec + cat_rec) if (cat_prec + cat_rec) > 0 else 0

    avg_fp = np.mean([s["eval_fp"] for s in cat_samples])
    avg_fn = np.mean([s["eval_fn"] for s in cat_samples])
    avg_mistake = np.mean([s["mistakenness"] for s in cat_samples])

    print(f"  {cat:23s} {n_imgs:7d} {cat_prec:10.3f} {cat_rec:10.3f} {cat_f1:8.3f} {avg_fp:8.1f} {avg_fn:8.1f} {avg_mistake:12.3f}")
    cat_stats.append({
        "category": cat, "n_images": n_imgs,
        "precision": cat_prec, "recall": cat_rec, "f1": cat_f1,
        "avg_fp": avg_fp, "avg_fn": avg_fn, "avg_mistakenness": avg_mistake,
    })

# Bar chart of per-category F1
cat_df = pd.DataFrame(cat_stats).sort_values("f1")

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#e74c3c" if f < 0.5 else "#f39c12" if f < 0.7 else "#27ae60" for f in cat_df["f1"]]
ax.barh(cat_df["category"], cat_df["f1"], color=colors, edgecolor="black", alpha=0.8)
ax.set_xlabel("F1 Score")
ax.set_title("Per-Scene Category Performance (Precision/Recall F1)")
overall_f1_val = 2 * overall_prec * overall_rec / (overall_prec + overall_rec) if (overall_prec + overall_rec) > 0 else 0
ax.axvline(x=overall_f1_val, color="black", linestyle="--", alpha=0.5,
           label=f"Overall F1={overall_f1_val:.3f}")
ax.legend()
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

### Think about it

- Which scene type has the **worst** performance? Why?
- Think about: object size, occlusion, camera distance, number of workers.
- Are the high-mistakenness scenes the same as the low-mAP scenes? What does that mean?

---
## Section 7: Data Curation Actions

Now that we understand our data quality issues, let us take action.

**Tiny label filtering** removes labels where either dimension is below a threshold.
These micro-boxes (often <20px at 640 resolution) are typically:
- False positives from the auto-labeler on background objects
- Truncated objects at image edges
- Objects too small for the model to learn from

In our reference experiments, filtering at 0.03 removed ~35% of labels and improved mAP50 by ~3%.
Your results may vary depending on which auto-labeler and prompts you used.


In [ ]:
# ── Cell 7a: Filter tiny labels ───────────────────────────────────────────
import subprocess

FILTERED_DIR = Path(str(DATASET_DIR) + "_filtered")
MIN_DIM = 0.03  # Try 0.02, 0.03, or 0.05 and compare label counts

print(f"Filtering tiny labels (min_dim={MIN_DIM})...")
print(f"  Input:  {DATASET_DIR}")
print(f"  Output: {FILTERED_DIR}")

result = subprocess.run(
    [
        sys.executable, str(DATA_DIR / "filter_tiny_labels.py"),
        "--input-dir", str(DATASET_DIR),
        "--output-dir", str(FILTERED_DIR),
        "--min-dim", str(MIN_DIM),
    ],
    capture_output=True, text=True,
    check=True,
)
print(result.stdout)


### Experiment: Try different thresholds

Change `MIN_DIM` above and re-run to see how different thresholds affect the label counts.
Try: `0.02`, `0.03`, `0.05`

- A lower threshold keeps more small objects (more data, more noise)
- A higher threshold is more aggressive (less noise, but may remove real small objects)

In [ ]:
# ── Cell 7b: Compare before/after filtering ───────────────────────────────

CLASS_COLORS_PLOT = {
    0: (0, 200, 0),      # hardhat -- green
    1: (220, 30, 30),     # person -- red
    2: (30, 100, 220),    # blue
}

def compare_before_after(image_path, label_before, label_after, class_names=None, figsize=(18, 7)):
    """Show an image with labels before and after filtering, side by side."""
    img = np.array(Image.open(image_path))
    h, w = img.shape[:2]

    def draw_labels(ax, label_path, title):
        ax.imshow(img)
        count = 0
        if Path(label_path).exists():
            for line in Path(label_path).read_text().strip().splitlines():
                parts = line.split()
                cls_id = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                color = np.array(CLASS_COLORS_PLOT.get(cls_id, (200, 200, 200))) / 255
                name = class_names.get(cls_id, f"cls_{cls_id}") if class_names else f"cls_{cls_id}"
                rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                     linewidth=2, edgecolor=color, facecolor="none")
                ax.add_patch(rect)
                ax.text(x1, y1-4, name, color=color, fontsize=7, fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.7))
                count += 1
        ax.set_title(f"{title} ({count} labels)", fontsize=11)
        ax.axis("off")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    draw_labels(ax1, label_before, "Before Filtering")
    draw_labels(ax2, label_after, "After Filtering")
    fig.suptitle(Path(image_path).name, fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Compare on a few validation images
if FILTERED_DIR.exists() and DATASET_DIR.exists():
    val_images = sorted((DATASET_DIR / "images" / "val").glob("*.*"))
    print(f"Comparing before/after on {min(3, len(val_images))} validation images:\n")
    for img_path in val_images[:3]:
        stem = img_path.stem
        compare_before_after(
            img_path,
            DATASET_DIR / "labels" / "val" / f"{stem}.txt",
            FILTERED_DIR / "labels" / "val" / f"{stem}.txt",
            class_names=CLASS_NAMES,
        )
else:
    print("Run the filtering step first (cell above).")

### Copilot Hook

Ask your AI copilot:

> "Based on the error analysis above, what other data curation steps could improve
> my labels? I found high mistakenness in X images and low uniqueness in Y images.
> The worst scene category is Z. What would you recommend?"

### Phase 3 Observations

Fill in your findings:
- Labels removed by filtering: ___%
- Most common label issue: ___
- Images with highest mistakenness: ___
- Worst scene category: ___ (mAP = ___)
- If you could only fix 5 images, which 5 and why? ___

---
## Section 8: Training

Now we train a fast YOLO26n student model on the curated (filtered) labels.

| Parameter | Value | Why |
|-----------|-------|-----|
| Model | `yolo26n.pt` | Nano variant -- fast inference, small footprint |
| Epochs | 100 | With patience=20 early stopping |
| Batch | 8 | Safe for A40 GPU memory |
| Image size | 640 | Standard YOLO training resolution |
| Workers | 0 | Avoids multiprocessing issues on HPC |

**Time estimate**: ~5-15 minutes on A40.

**Short on time?** Pre-baked weights are available at `data/ppe_results/`.

In [ ]:
# ── Cell 8a: Train YOLO26n on curated labels ──────────────────────────────
# ============================================================
# TRAINING CONFIGURATION
# ============================================================
TRAIN_DATASET = FILTERED_DIR if FILTERED_DIR.exists() else DATASET_DIR
TRAIN_YAML = TRAIN_DATASET / "dataset.yaml"
if not TRAIN_YAML.exists():
    TRAIN_YAML = TRAIN_DATASET / "data.yaml"
EPOCHS = 100
PATIENCE = 20
BATCH = 8
IMGSZ = 640

# MPS has intermittent tensor shape mismatch bugs in Ultralytics loss
# computation (TAL module). Use CUDA on HPC, fall back to CPU otherwise.
# For 91 images, CPU training is fast enough (~5-10 min).
TRAIN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EXPERIMENT_NAME = "nb02_curated"
# ============================================================

assert TRAIN_YAML.exists(), f"Dataset YAML not found at {TRAIN_YAML}"
print(f"Training dataset: {TRAIN_DATASET}")
print(f"Config: epochs={EPOCHS}, patience={PATIENCE}, batch={BATCH}, imgsz={IMGSZ}, device={TRAIN_DEVICE}")

from ultralytics import YOLO

train_model = YOLO("yolo26n.pt")
train_results = train_model.train(
    data=str(TRAIN_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,
    batch=BATCH,
    imgsz=IMGSZ,
    device=TRAIN_DEVICE,
    workers=0,
    project=str(DATA_DIR / "ppe_results"),
    name=EXPERIMENT_NAME,
)

print(f"\nTraining complete!")
print(f"Best weights: {train_results.save_dir}/weights/best.pt")

### While training runs...

Look back at your **Data Quality Dashboard** (Section 5). Based on what you found, would you:
- (a) Collect more data from the worst scene categories?
- (b) Fix specific labels flagged by mistakenness?
- (c) Try different auto-labeling prompts?
- (d) Change the filter threshold?

There is no single right answer. The point is to have a **hypothesis** about what
matters most, and then test it.

### Copilot Hook

Ask your AI copilot:

> "My training is running. While I wait, help me think through: my mAP was X,
> the worst scene category was Y with mAP Z, and Z% of labels had high mistakenness.
> What is the single highest-leverage change I could make for the next iteration?"

In [ ]:
# ── Cell 8b: Plot training curves ─────────────────────────────────────────

def plot_training_curves(results_csv, figsize=(16, 5)):
    """Plot training curves from Ultralytics results.csv."""
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    # Training loss
    loss_cols = [c for c in df.columns if "loss" in c.lower() and "val" not in c.lower()]
    ax = axes[0]
    for col in loss_cols:
        ax.plot(df["epoch"], df[col], label=col, alpha=0.8)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # Validation loss
    val_loss_cols = [c for c in df.columns if "loss" in c.lower() and "val" in c.lower()]
    ax = axes[1]
    for col in val_loss_cols:
        ax.plot(df["epoch"], df[col], label=col, alpha=0.8)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Validation Loss")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # mAP
    ax = axes[2]
    map_cols = [c for c in df.columns if "map" in c.lower() or "mAP" in c]
    for col in map_cols:
        ax.plot(df["epoch"], df[col], label=col, alpha=0.8, linewidth=2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("mAP")
    ax.set_title("Validation mAP")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Best metrics
    if map_cols:
        best_col = [c for c in map_cols if "50" in c and "95" not in c]
        if best_col:
            best_epoch = df[best_col[0]].idxmax()
            best_val = df[best_col[0]].max()
            print(f"\nBest mAP50: {best_val:.4f} at epoch {int(df['epoch'].iloc[best_epoch])}")


# Find results.csv -- from our training or pre-baked
results_dir = DATA_DIR / "ppe_results" / EXPERIMENT_NAME
results_csv = results_dir / "results.csv"

if results_csv.exists():
    plot_training_curves(results_csv)
else:
    # Fallback to any pre-baked results
    prebaked = list((DATA_DIR / "ppe_results").glob("*/results.csv"))
    if prebaked:
        print(f"Using pre-baked results: {prebaked[-1].parent.name}")
        plot_training_curves(prebaked[-1])
    else:
        print("No results.csv found. Train a model first or check data/ppe_results/")

---
## Section 9: Post-Training Evaluation Loop

Now let us evaluate the freshly trained model using the same IoU-based evaluation,
and compare with the baseline results from the beginning of this notebook.

In [ ]:
# ── Cell 9a: Evaluate the newly trained model ─────────────────────────────

# Find the best weights from the training run
new_weights = DATA_DIR / "ppe_results" / EXPERIMENT_NAME / "weights" / "best.pt"
if not new_weights.exists():
    # Fallback: use the original weights
    new_weights = BEST_WEIGHTS
    print(f"Using original weights: {new_weights}")
else:
    print(f"Using newly trained weights: {new_weights}")

# Load new model
new_student = YOLO(str(new_weights))

# Run inference on every sample and store as predictions_v2
print(f"\nRunning inference with new model on {len(dataset)} images...")
for i, sample in enumerate(dataset):
    results = new_student.predict(sample["filepath"], verbose=False, device=DEVICE)
    preds = []
    if results and results[0].boxes is not None:
        for box in results[0].boxes:
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            xyxy = box.xyxy[0].tolist()
            preds.append({"cls": cls_id, "bbox": xyxy, "conf": conf})
    sample["predictions_v2"] = preds
    if (i + 1) % 20 == 0 or (i + 1) == len(dataset):
        print(f"  {i+1}/{len(dataset)} images processed")

# Evaluate predictions_v2 using IoU matching
agg_v2 = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
for sample in dataset:
    eval_result = evaluate_image(
        sample["gt_labels"], sample["predictions_v2"],
        sample["img_w"], sample["img_h"],
        CLASS_NAMES, iou_threshold=IOU_THRESHOLD,
    )
    sample["eval_v2_tp"] = eval_result["tp"]
    sample["eval_v2_fp"] = eval_result["fp"]
    sample["eval_v2_fn"] = eval_result["fn"]
    for cls_id, counts in eval_result["per_class"].items():
        agg_v2[cls_id]["tp"] += counts["tp"]
        agg_v2[cls_id]["fp"] += counts["fp"]
        agg_v2[cls_id]["fn"] += counts["fn"]

# Also run official mAP via model.val()
print("\nRunning Ultralytics model.val() for official mAP (after curation)...")
val_results_v2 = new_student.val(
    data=str(yaml_path),
    device=DEVICE,
    verbose=False,
)
mAP_v2 = float(val_results_v2.box.map50)
mAP_v2_95 = float(val_results_v2.box.map)

# Compute aggregate metrics
v2_tp = sum(m["tp"] for m in agg_v2.values())
v2_fp = sum(m["fp"] for m in agg_v2.values())
v2_fn = sum(m["fn"] for m in agg_v2.values())
v2_prec = v2_tp / (v2_tp + v2_fp) if (v2_tp + v2_fp) > 0 else 0
v2_rec = v2_tp / (v2_tp + v2_fn) if (v2_tp + v2_fn) > 0 else 0
v2_f1 = 2 * v2_prec * v2_rec / (v2_prec + v2_rec) if (v2_prec + v2_rec) > 0 else 0

delta_mAP = mAP_v2 - baseline_mAP50

print(f"\n{'='*60}")
print(f"  COMPARISON: Baseline vs Curated Training")
print(f"{'='*60}")
print(f"  Baseline mAP@0.5:         {baseline_mAP50:.4f}")
print(f"  After curation mAP@0.5:   {mAP_v2:.4f}")
print(f"  Delta:                     {delta_mAP:+.4f} ({'+' if delta_mAP >= 0 else ''}{delta_mAP/max(baseline_mAP50, 0.001)*100:.1f}%)")
print(f"{'='*60}")

print(f"\nPer-class report (after curation, IoU >= {IOU_THRESHOLD}):")
print("-" * 70)
print(f"  {'Class':<15} {'TP':>6} {'FP':>6} {'FN':>6} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"  {'-'*15} {'-'*6} {'-'*6} {'-'*6} {'-'*10} {'-'*10} {'-'*10}")

for cls_id in sorted(agg_v2.keys()):
    m = agg_v2[cls_id]
    tp, fp, fn = m["tp"], m["fp"], m["fn"]
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    cls_name = CLASS_NAMES.get(cls_id, f"cls_{cls_id}")
    print(f"  {cls_name:<15} {tp:>6} {fp:>6} {fn:>6} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}")

print(f"  {'-'*15} {'-'*6} {'-'*6} {'-'*6} {'-'*10} {'-'*10} {'-'*10}")
print(f"  {'TOTAL':<15} {v2_tp:>6} {v2_fp:>6} {v2_fn:>6} {v2_prec:>10.4f} {v2_rec:>10.4f} {v2_f1:>10.4f}")

### Think about it

- Did your mAP improve? By how much?
- What was the biggest factor: the auto-labeler choice, the prompt, or the filtering?
- Which class improved more: `hardhat` or `person`? Why?
- If you had another iteration, what would you change?

### Copilot Hook

Ask your AI copilot:

> "My model went from mAP X to Y after data curation. The per-class results show...
> What would be the most impactful next step: (a) more data, (b) better auto-labeling
> prompts, (c) a larger model, or (d) different training hyperparameters? Explain why."

---
## Section 10: Phase 3-4 Reflection

### Observations

Fill in:
- Baseline mAP50: ___
- After curation mAP50: ___
- Training time: ___ minutes
- Best epoch: ___ (out of how many?)
- Did early stopping trigger? ___
- Worst scene category: ___
- Number of suspicious labels found: ___

### Key Takeaways

1. **Data-centric > model-centric.** We did not change the model architecture or hyperparameters.
   We improved the DATA, and the model improved.

2. **The student audits the teacher.** Mistakenness leverages the trained student to find
   where the auto-labeler made mistakes.

3. **Evaluation beyond mAP.** A single number hides critical information. Per-sample and
   per-scene analysis reveals exactly where to act.

4. **Embeddings reveal structure.** UMAP projections show whether errors cluster in
   specific visual conditions.

### What would you do differently in the next iteration?

___

---

**Next**: Open `03_evaluate_and_deploy.ipynb` to evaluate the model and build a compliance system.

In [ ]:
# ── Cleanup ───────────────────────────────────────────────────────────────
# No database cleanup needed -- all data is in-memory Python dicts.
# The dataset list, predictions, and evaluation metrics are all stored
# in the `dataset` variable (a plain list of dicts).

print(f"Notebook complete!")
print(f"  Dataset:  {len(dataset)} validation samples")
print(f"  Baseline mAP@0.5:       {baseline_mAP50:.4f}")
print(f"  After curation mAP@0.5: {mAP_v2:.4f}")
print(f"\nAll data stored in `dataset` (list of dicts) -- no cleanup required.")